In [ ]:
# Practice Problems Day 14

### Author: Joanna Bieri (solutions)

## Introduction to Machine Learning

#### University of Redlands - DATA 301
#### Prof: Joanna Bieri [joanna_bieri@redlands.edu](mailto:joanna_bieri@redlands.edu)
#### Day 14 Practice - SOLUTIONS
#### [Class Website](https://joannabieri.com/machine_learning.html)

---

**DRAFT - not yet edited by Joanna**

**These are practice problems. They are not graded and you do not turn them in.** They are here so you can practice today's code before the weekly homework. Weekly Homework 7 assumes you have done them. Plan on about 45 minutes. Each training run takes a minute or two on a laptop.

**These are my solutions.** Try every problem yourself first, then come here to check your work. There is more than one right way to write the code, and your words will be different from mine, that is fine! If your numbers do not match, look for the difference before you look at my code.

**Reading:** Geron chapter 12, **Convolutional Layers**, **Pooling Layers** and **CNN Architectures**.

GOALS:

1. Build a small CNN with `nn.Conv2d`, `nn.MaxPool2d` and `nn.Flatten`.
2. Work out the shapes, so the last `nn.Linear` layer gets the right size.
3. Explain a convolution in words.

**How much of this should you write yourself?** The data cell is given. Problem 1 gives you the training loop, you write the layers. Problem 2 lists the steps. Problem 3 is words only. The **Optional exploration** at the end is extra, only if you are curious.

**Save your work.** Commit and push when you are done, so you have your code for the weekly homework and the exam:

```bash
git add .
git commit -m "Day 14 practice"
git push
```

---

In [ ]:
# RUN THIS. The Fashion MNIST pictures from the notes, as tensors.
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torchvision
from torch.utils.data import TensorDataset, DataLoader

fashion = torchvision.datasets.FashionMNIST(root="data", train=True, download=True)
class_names = fashion.classes

images = fashion.data[:12000].float() / 255     # pixels from 0 to 1
labels = fashion.targets[:12000]
images = images.reshape(-1, 1, 28, 28)          # (pictures, channels, height, width)

X_train = images[:10000]
y_train = labels[:10000]
X_valid = images[10000:]
y_valid = labels[10000:]
train_data = TensorDataset(X_train, y_train)

print("training pictures:  ", X_train.shape)
print("validation pictures:", X_valid.shape)

## Problem 1: A CNN with one round of convolution and pooling

**What to copy, what to write:** The training loop is given. Write the layers the comments ask for.

**1a.** Build a CNN with **one** round: `nn.Conv2d` with 1 channel in, **8** filters, `kernel_size=3` and `padding=1`, then `nn.ReLU()`, then `nn.MaxPool2d(2)`, then `nn.Flatten()`, then an `nn.Linear` layer that gives 10 scores. Train it for 5 epochs.

**1b.** What is the shape of each picture after the pooling layer (channels, height, width)? Use it to explain the number you put in the `nn.Linear` layer.

**1c.** Compare your final validation accuracy to the notes: the MLP got 0.858 and the CNN with two rounds got 0.877.

In [ ]:
# 1a: one round of convolution and pooling
torch.manual_seed(42)
model = nn.Sequential(
    nn.Conv2d(1, 8, kernel_size=3, padding=1),     # 8 x 28 x 28
    nn.ReLU(),
    nn.MaxPool2d(2),                               # 8 x 14 x 14
    nn.Flatten(),                                  # 8 * 14 * 14 = 1568 numbers
    nn.Linear(8 * 14 * 14, 10))                    # 10 scores

# The batches, the loss and the optimizer
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# The training loop
n_epochs = 5
for epoch in range(n_epochs):
    model.train()
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        scores = model(X_batch)
        loss = loss_fn(scores, y_batch)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        y_pred = model(X_valid).argmax(dim=1)
        accuracy = (y_pred == y_valid).float().mean().item()
    print("epoch", epoch + 1, "  valid accuracy:", round(accuracy, 4))

model_1 = model                                    # save it so we can count its numbers in Problem 2

**1a.** The first layer is `nn.Conv2d(1, 8, kernel_size=3, padding=1)` and the last is `nn.Linear(8 * 14 * 14, 10)`. After 5 epochs the validation accuracy is 0.8435.

**1b.** After the convolution each picture is 8 by 28 by 28 (8 filters, and `padding=1` keeps it 28 by 28). Pooling halves the height and width, so it is **8 by 14 by 14**. `nn.Flatten()` makes that one row of $8 \times 14 \times 14 = 1568$ numbers, so the `nn.Linear` layer needs 1568 inputs.

**1c.** 0.8435 is a little behind both the MLP (0.858) and the notes' CNN with two rounds (0.877). One round of convolution with only 8 filters is not enough to beat the MLP here. Problem 2 adds the second round.

---

## Problem 2: Add a second round (steps outlined)

**What to copy, what to write:** Copy your Problem 1 cell and change the model. The training loop stays the same.

**2a.** Add a second round after the first pooling layer: `nn.Conv2d` with 8 channels in and **16** filters, `kernel_size=3`, `padding=1`, then `nn.ReLU()`, then `nn.MaxPool2d(2)`. Fix the size of the `nn.Linear` layer and train it for 5 epochs.

**2b.** Count the numbers to learn in your Problem 1 model and your Problem 2 model. Which one has more? Which one did better?

In [ ]:
# 2a: two rounds of convolution and pooling
torch.manual_seed(42)
model = nn.Sequential(
    nn.Conv2d(1, 8, kernel_size=3, padding=1),     # 8 x 28 x 28
    nn.ReLU(),
    nn.MaxPool2d(2),                               # 8 x 14 x 14
    nn.Conv2d(8, 16, kernel_size=3, padding=1),    # 16 x 14 x 14
    nn.ReLU(),
    nn.MaxPool2d(2),                               # 16 x 7 x 7
    nn.Flatten(),                                  # 16 * 7 * 7 = 784 numbers
    nn.Linear(16 * 7 * 7, 10))

# The batches, the loss and the optimizer
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# The training loop
n_epochs = 5
for epoch in range(n_epochs):
    model.train()
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        scores = model(X_batch)
        loss = loss_fn(scores, y_batch)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        y_pred = model(X_valid).argmax(dim=1)
        accuracy = (y_pred == y_valid).float().mean().item()
    print("epoch", epoch + 1, "  valid accuracy:", round(accuracy, 4))

model_2 = model

In [ ]:
# 2b: count the numbers to learn
n_1 = 0
for p in model_1.parameters():
    n_1 = n_1 + p.numel()

n_2 = 0
for p in model_2.parameters():
    n_2 = n_2 + p.numel()

print("one round: ", n_1)
print("two rounds:", n_2)

**2a.** After the second pooling layer each picture is **16 by 7 by 7**, so the `nn.Linear` layer is `nn.Linear(16 * 7 * 7, 10)`, 784 inputs. After 5 epochs the validation accuracy is 0.86.

**2b.** The one round model has **15,770** numbers to learn and the two round model has **9,098**. Most of them are in the last `nn.Linear` layer: one round leaves 1568 numbers going into it, two rounds only 784, because the second pooling layer halves the height and width again. So the two round model has fewer numbers to learn, and it did better (0.86 against 0.8435).

---

## Problem 3: In words

**3a.** Explain what a convolution layer does to a friend in DATA 101 who has never seen one. Use the words filter and feature map. Three or four sentences, no code.

**3b.** A classmate gets this error when they train their CNN:

`RuntimeError: mat1 and mat2 shapes cannot be multiplied (32x1568 and 784x10)`

Their last layer is `nn.Linear(784, 10)`. What went wrong, and what should the layer be?

**3a.** Your answer will be different, here is one way to say it:

A convolution layer slides a small grid of numbers, called a filter, over the picture. At every spot it multiplies the pixels under the filter by the filter's numbers and adds them up, and that gives one number for that spot. All those numbers together make a feature map, a picture of where the filter found what it is looking for, like an edge. The layer has many filters, and the network learns their numbers during training.

**3b.** The 1568 is the number of numbers coming out of `nn.Flatten()`, and the `nn.Linear` layer was built for 784. The shapes have to match. The layer should be `nn.Linear(1568, 10)`, for example `nn.Linear(32 * 7 * 7, 10)` if the last convolution layer has 32 filters and the pictures are 7 by 7 after pooling.

---

## Optional exploration

Only if you are curious. Nothing here is needed for the weekly homework.

**A.** Use your Problem 2 model to find 5 validation pictures it gets wrong. Plot them with their real class and the predicted class as the title. Would you have gotten them right?

**B.** Add `nn.BatchNorm2d(8)` right after the first `nn.Conv2d` and `nn.BatchNorm2d(16)` right after the second. Does it change the accuracy on this small CNN?

In [ ]:
# Optional A: 5 validation pictures the two round CNN gets wrong
model_2.eval()
with torch.no_grad():
    y_pred = model_2(X_valid).argmax(dim=1)

wrong = torch.nonzero(y_pred != y_valid).flatten()     # the positions where the prediction is wrong
print("wrong:", len(wrong), "out of", len(y_valid))

fig, axes = plt.subplots(1, 5, figsize=(12, 3))
for k in range(5):
    i = wrong[k]
    axes[k].imshow(X_valid[i, 0], cmap="gray_r")
    axes[k].set_title("real: " + class_names[y_valid[i]] + "\npredicted: " + class_names[y_pred[i]], fontsize=9)
    axes[k].axis("off")
plt.show()

In [ ]:
# Optional B: the two round CNN with batch normalization
torch.manual_seed(42)
model = nn.Sequential(
    nn.Conv2d(1, 8, kernel_size=3, padding=1),     # 8 x 28 x 28
    nn.BatchNorm2d(8),
    nn.ReLU(),
    nn.MaxPool2d(2),                               # 8 x 14 x 14
    nn.Conv2d(8, 16, kernel_size=3, padding=1),    # 16 x 14 x 14
    nn.BatchNorm2d(16),
    nn.ReLU(),
    nn.MaxPool2d(2),                               # 16 x 7 x 7
    nn.Flatten(),                                  # 16 * 7 * 7 = 784 numbers
    nn.Linear(16 * 7 * 7, 10))

# The batches, the loss and the optimizer
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# The training loop
n_epochs = 5
for epoch in range(n_epochs):
    model.train()
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        scores = model(X_batch)
        loss = loss_fn(scores, y_batch)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        y_pred = model(X_valid).argmax(dim=1)
        accuracy = (y_pred == y_valid).float().mean().item()
    print("epoch", epoch + 1, "  valid accuracy:", round(accuracy, 4))

**A.** The two round CNN gets 280 of the 2000 validation pictures wrong. 176 of those 280 mistakes are shirts, T-shirts, pullovers and coats mixed up with each other. At 28 by 28 pixels some of them are hard for a person too!

**B.** With batch normalization the final accuracy is 0.87, against 0.86 without it. That is a small bump on a CNN this small. Batch normalization really matters in very deep CNNs.